# Indexing Algorithms

**Module:** 02 — Vector Databases

HNSW, IVF family, PQ, and compression.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain HNSW knobs
- Describe IVF/Flat/PQ
- Estimate compression tradeoffs


## HNSW

**Definition.** **HNSW** multi-layer proximity graph for greedy routing.

**Why it matters.** Strong default for in-memory ANN.

**How it works.** M neighbors/layer; efSearch beam at query.

**Intuition.** Highways then streets.

**Common pitfalls.**
- ef too low
- M too high → RAM

**When to use.** RAM-rich interactive search.

```mermaid
flowchart TD
 L2-->L1-->L0; Q-->L2
```


In [ ]:
print({'M':16,'efConstruction':100,'efSearch':64})


In [ ]:
import numpy as np
for ef in [16,64,256]:
    print(ef, round(0.995*(1-np.exp(-ef/40)),3), round(2+0.05*ef,2))


In [ ]:
import json
print(json.dumps({'hnsw_config':{'m':16,'ef_construct':100}},indent=2))


### Try it yourself — HNSW

1. Sweep ef for recall≥0.98.


## IVF (Inverted File)

**Definition.** Partition into nlist clusters; probe nprobe lists.

**Why it matters.** Fewer comparisons; FAISS classic.

**How it works.** k-means assign → probe centroids → scan lists.

**Intuition.** Phone book by city.

**Common pitfalls.**
- Bad nlist
- Tiny nprobe

**When to use.** Large N / FAISS stacks.


In [ ]:
import numpy as np
rng=np.random.default_rng(0); X=rng.normal(size=(3000,16)).astype('float32'); nlist=32
cent=X[rng.choice(len(X),nlist,False)].copy(); assign=np.argmin(((X[:,None,:]-cent[None,:,:])**2).sum(-1),1)
lists=[np.where(assign==i)[0] for i in range(nlist)]; q=X[0]; probe=np.argsort(((cent-q)**2).sum(1))[:4]
cand=np.concatenate([lists[i] for i in probe]); print(cand[np.argsort(-(X[cand]@q))[:5]])


In [ ]:
for p in [1,4,16]: print(p, f'{p/32:.2%}')


In [ ]:
import math
print([(n,int(math.sqrt(n))) for n in [1e5,1e6,5e7]])


### Try it yourself — IVF (Inverted File)

1. Propose nlist/nprobe for N=5e6.


## IVF Flat

**Definition.** IVF with full-precision vectors in lists.

**Why it matters.** Higher quality than PQ when RAM allows.

**How it works.** Probe lists; exact distances inside.

**Intuition.** City index + full photos.

**Common pitfalls.**
- RAM underestimation

**When to use.** High recall, RAM available.


In [ ]:
n,d=10_000_000,768; print('flat',n*d*4/1024**3,'pq96',n*96/1024**3)


In [ ]:
print({'IVF Flat':'IVF4096,Flat','IVF PQ':'IVF4096,PQ64'})


In [ ]:
print({'prefer_flat':['RAM OK','need recall']})


### Try it yourself — IVF Flat

1. When move Flat→PQ?


## IVF PQ

**Definition.** IVF + Product Quantization codes.

**Why it matters.** Billion-scale under RAM caps.

**How it works.** Codes in lists; distance tables; often rescore.

**Intuition.** Thumbnails then zoom.

**Common pitfalls.**
- Aggressive PQ cliffs
- No rescoring

**When to use.** Memory-bound large N.


In [ ]:
def pq_bytes(dim,m,bits=8):
    assert dim%m==0; return m*(bits//8)
print([(m,pq_bytes(768,m)) for m in [16,32,64,96]])


In [ ]:
import numpy as np
rng=np.random.default_rng(0); cb=rng.normal(size=(4,4,2)); q=rng.normal(size=(4,2))
tables=np.linalg.norm(cb-q[:,None,:],axis=2); code=np.array([0,2,1,3])
print(float(tables[np.arange(4),code].sum()))


In [ ]:
print({'M':100,'k':10,'rescore':True})


### Try it yourself — IVF PQ

1. Config for 100M×768 in ~64GiB codes.


## Product Quantization (PQ)

**Definition.** Split into m subspaces; quantize each.

**Why it matters.** Compression + fast approx distance.

**How it works.** Codebooks per subspace; sum table distances.

**Intuition.** Template parts reconstruct approximately.

**Common pitfalls.**
- m does not divide dim
- Bad training sample

**When to use.** With IVF or standalone compress.


In [ ]:
import numpy as np
rng=np.random.default_rng(0); dim,m,ksub=8,4,4; dsub=dim//m; train=rng.normal(size=(500,dim))
cb=[train[:,j*dsub:(j+1)*dsub][rng.choice(500,ksub,False)] for j in range(m)]
def enc(v): return [int(np.argmin(np.linalg.norm(cb[j]-v[j*dsub:(j+1)*dsub],axis=1))) for j in range(m)]
print(enc(train[0]))


In [ ]:
print(f'compression {768*4/96:.1f}x')


In [ ]:
print(['dim%m==0','representative train','eval recall'])


### Try it yourself — Product Quantization (PQ)

1. Why asymmetric distance helps.


## Vector Compression

**Definition.** PQ/int8/float16/binary to cut bytes.

**Why it matters.** Cost/RAM at scale.

**How it works.** Codec → query compressed → rescore top-M.

**Intuition.** JPEG for embeddings.

**Common pitfalls.**
- No eval
- No rescoring path

**When to use.** When working set > RAM comfort.

| Method | Savings |
|--------|---------|
| float16 | ~2× |
| int8 | ~4× |
| PQ | 8–32× |


In [ ]:
import numpy as np
v=np.random.default_rng(0).normal(size=768).astype('float32'); m=np.max(np.abs(v))+1e-9
q=np.clip(v/m*127,-127,127).astype('int8'); vhat=q.astype('float32')/127*m
print(round(float(np.dot(v,vhat)/(np.linalg.norm(v)*np.linalg.norm(vhat)+1e-9)),4), v.nbytes, q.nbytes)


In [ ]:
print('rescore shortlist before final top-k')


In [ ]:
print('cost flat',200*0.5,'pq',25*0.5)


### Try it yourself — Vector Compression

1. Compression+rescore plan for 50M on 128GiB.


## Glossary

- **efSearch**: HNSW query beam
- **nprobe**: IVF lists scanned
- **rescoring**: Exact distances on shortlist


## Summary & Key Takeaways

- HNSW for RAM-rich recall.
- IVF scales lists; PQ compresses.
- Measure recall after compression.

### Practice

Sweep ef or nprobe with recall@10 and p95 gates.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
